# 11.0 KI-Kickstart: Wie funktionieren Sprachmodelle?

Dieses Notebook gehört zur Kursseite **11.0**. Führe die Zellen der Reihe nach mit `Umschalt + Enter` aus.
Vergleiche jede Ausgabe mit dem Kommentar `# erwartet: ...`.

## 1. Tokenisierung (Beispiel 1)
An jeder Stelle wird das längste Stück gewählt, das im Vokabular steht.

In [ ]:
vokabular = ["informat", "inform", "iker", "ik", "er", "in", "atik", "ung"]
for buchstabe in "abcdefghijklmnopqrstuvwxyzäöüß":
    vokabular.append(buchstabe)        # IDs 8 bis 37

def tokenisiere(wort, vokabular):
    """Zerlegt wort: an jeder Stelle das längste passende Vokabular-Stück."""
    tokens = []
    pos = 0
    while pos < len(wort):
        laenge = len(wort) - pos
        gefunden = False
        while laenge > 0 and not gefunden:
            stueck = wort[pos:pos + laenge]
            if stueck in vokabular:
                tokens.append(stueck)
                pos = pos + laenge
                gefunden = True
            else:
                laenge = laenge - 1
        if not gefunden:
            tokens.append("<unbekannt>")
            pos = pos + 1
    return tokens

def token_ids(tokens, vokabular):
    ids = []
    for t in tokens:
        if t in vokabular:
            ids.append(vokabular.index(t))
        else:
            ids.append(-1)
    return ids

print(tokenisiere("informatikerin", vokabular))          # erwartet: ['informat', 'iker', 'in']
print(token_ids(["informat", "iker", "in"], vokabular))  # erwartet: [0, 2, 5]
print(tokenisiere("spielerin", vokabular))               # erwartet: ['s', 'p', 'i', 'e', 'l', 'er', 'in']
print(tokenisiere("", vokabular))                        # erwartet: []
print(tokenisiere("c3po", vokabular))                    # erwartet: ['c', '<unbekannt>', 'p', 'o']

**Aufgabe 11.0.1:** Prüfe hier deine Handrechnung für „informatik“, „formatierung“ und „informatikerinnen“.

In [ ]:
for wort in ["informatik", "formatierung", "informatikerinnen"]:
    tokens = tokenisiere(wort, vokabular)
    print(wort, tokens, token_ids(tokens, vokabular))

## 2. Embeddings und Kosinus-Ähnlichkeit (Beispiel 2)

In [ ]:
import math

def kosinus(a, b):
    skalar = 0
    laenge_a = 0
    laenge_b = 0
    for i in range(len(a)):
        skalar = skalar + a[i] * b[i]
        laenge_a = laenge_a + a[i] ** 2
        laenge_b = laenge_b + b[i] ** 2
    return skalar / (math.sqrt(laenge_a) * math.sqrt(laenge_b))

hund  = [0.8, 0.6, 0.1]
katze = [0.7, 0.7, 0.2]
auto  = [0.1, 0.2, 0.9]
print(round(kosinus(hund, katze), 3))   # erwartet: 0.985
print(round(kosinus(hund, auto), 3))    # erwartet: 0.311
print(round(kosinus(hund, hund), 3))    # erwartet: 1.0

**Aufgabe 11.0.8:** Ergänze den Vektor `maus = [0.6, 0.5, 0.3]` und berechne die Ähnlichkeiten zu Hund, Katze und Auto.

In [ ]:
# Deine Lösung zu 11.0.8

## 3. Softmax und Temperatur (Beispiel 3)

In [ ]:
import random

def softmax(logits, temperatur=1.0):
    exps = []
    for z in logits:
        exps.append(math.exp(z / temperatur))
    summe = sum(exps)
    wahrscheinlichkeiten = []
    for e in exps:
        wahrscheinlichkeiten.append(e / summe)
    return wahrscheinlichkeiten

kandidaten = ["Dresden", "Leipzig", "Chemnitz", "schön"]
logits = [4.0, 2.5, 1.5, 1.0]

for T in [0.5, 1.0, 2.0]:
    p = softmax(logits, T)
    gerundet = []
    for wert in p:
        gerundet.append(round(wert, 3))
    print(T, gerundet)
# erwartet:
# 0.5 [0.944, 0.047, 0.006, 0.002]
# 1.0 [0.738, 0.165, 0.061, 0.037]
# 2.0 [0.505, 0.238, 0.145, 0.113]

p = softmax(logits, 1.0)
print(kandidaten[p.index(max(p))])                  # erwartet: Dresden
print(random.choices(kandidaten, weights=p, k=10))  # zufällig, meist 'Dresden'
print(round(-math.log(p[0]), 3))                    # erwartet: 0.304
print(round(-math.log(p[1]), 3))                    # erwartet: 1.804

**Experiment:** Ziehe 1000-mal mit `random.choices` bei T = 2 und zähle, wie oft „schön“ vorkommt. Erwartet sind etwa 113.

**Aufgabe 11.0.2:** Prüfe deine Handrechnung (Kaffee 2,0 · Tee 1,0 · Kakao 0,5 · Suppe −1,0).

In [ ]:
# Experiment und Kontrolle zu 11.0.2

## 4. Aufgabe 11.0.4: Top-k-Auswahl
Schreibe die Funktion `top_k` und prüfe sie mit den Testfällen.

In [ ]:
def top_k(kandidaten, wahrscheinlichkeiten, k):
    # deine Lösung
    pass

p = softmax([4.0, 2.5, 1.5, 1.0], 1.0)
for k in [2, 1, 4, 10, 0]:
    print(k, top_k(kandidaten, p, k))
# erwartet (gerundet): k=2 -> [('Dresden', 0.818), ('Leipzig', 0.182)], k=1 -> [('Dresden', 1.0)], k=0 -> []

## 5. Aufgabe 11.0.5: Pauls Tokenizer
Sage die Ausgaben **zuerst ohne Ausführen** voraus und notiere sie. Führe die Zelle erst danach aus.

In [ ]:
def tokenisiere_paul(wort, vokabular):
    tokens = []
    pos = 0
    while pos < len(wort):
        laenge = 1
        gefunden = False
        while laenge <= len(wort) - pos and not gefunden:
            stueck = wort[pos:pos + laenge]
            if stueck in vokabular:
                tokens.append(stueck)
                pos = pos + laenge
                gefunden = True
            else:
                laenge = laenge + 1
        if not gefunden:
            tokens.append("<unbekannt>")
            pos = pos + 1
    return tokens

for wort in ["informatikerin", "c3po", ""]:
    print(repr(wort), tokenisiere_paul(wort, vokabular), tokenisiere(wort, vokabular))